## **Install & Import necessary libraries**

In [ ]:
import pandas as pd
import spacy
from spacy import displacy
import datetime


## **Mount the dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# df = pd.read_json('/content/drive/MyDrive/ITE Elective 4/NER-Exercise/News_Category_Dataset_v3 2.json', lines=True)
df = pd.read_csv('/content/drive/MyDrive/ITE Elective 4/NER-Exercise/sample_text.csv')

df.head()

Mounted at /content/drive


,text
0,How to Manage Your Personal Brand. Make no mis...


## **Manual Tagging/Annotation**

In [ ]:
nlp = spacy.load("en_core_web_sm")

if __name__ == "__main__":
  print("\n--- Named Entity Recognition with spaCy ---")

  if 'ner' in nlp.pipe_names:
    ner_labels = nlp.get_pipe('ner').labels
    print("\n--- Named Entity Recognition with spaCy ---")
    print(", ".join(sorted(ner_labels)))
  else:
    print("\nNER pipeline component not found in the loaded model.")

print("\nEnter the text you want to analyze (type 'quit' to exit):")
output_counter = 0



--- Named Entity Recognition with spaCy ---

--- Named Entity Recognition with spaCy ---
CARDINAL, DATE, EVENT, FAC, GPE, LANGUAGE, LAW, LOC, MONEY, NORP, ORDINAL, ORG, PERCENT, PERSON, PRODUCT, QUANTITY, TIME, WORK_OF_ART

Enter the text you want to analyze (type 'quit' to exit):


In [ ]:
while True:
  input_text = input("\nEnter text to process (enter qui to sti): ")

  if input_text.lower() == 'quit':
    break
  elif not input_text.strip():
    print("Invalid input: Please provide a non-empty string for processing.")
    continue

  doc = nlp(input_text)

  print("\n--- Detected Named Entities ---")
  if doc.ents:
      for ents in doc.ents:
        print(f"Text: '{ents.text}' | Label: {ents.label} ({spacy.explain(ents.label_)}) ")
  else:
    print("No named entities detected in the text.")



Enter text to process (enter qui to sti): How to Manage Your Personal Brand. Make no mistake: If you have a Facebook account, an Instagram page, a Twitter profile, you are a brand. Every time you upload a photo, add a link, or post an update, you're putting into the world another idea of yourself and what you stand for. It Looks Like Uber's Winning Its War With New York. Grab the popcorn. The Progressive Promise of Today's Technology. A digital policy for the new century, tailored not just to the moment but for the future, is vital if we are to unleash economic growth, shared prosperity, and the full potential of technology for citizens and consumers. But such a policy architecture requires a new consensus -- on privacy, on security, on customer protections, on growth and mobility. U.S. Lawmakers Join Demand For Puerto Rico Governor’s Resignation. Puerto Ricans staged week-long protests calling on Ricardo Rossello to quit over leaked misogynistic and homophobic messages. As Florida Go

## **Automated Entity Recognition with spaCy**

In [ ]:
import pandas as pd
import spacy

def extract_entities(df, text_column="text"):
    """Extract entities from a DataFrame column using spaCy NER."""
    nlp = spacy.load("en_core_web_sm")
    records = []

    if text_column not in df.columns:
        raise ValueError(f"Column '{text_column}' not found in DataFrame.")

    for i, row in df.iterrows():
        text = str(row[text_column]).strip()
        doc = nlp(text)
        for ent in doc.ents:
            records.append({
                "Sample_ID": i + 1,
                "Original_Text": text,
                "Entity": ent.text,
                "Label": ent.label_
            })
    return pd.DataFrame(records)


def preview_entities(ner_df):
    """Print formatted entities grouped by Sample_ID."""
    print("\n--- Extracted Entities (Grouped by Sample) ---\n")
    if ner_df.empty:
        print("⚠️ No entities were extracted.")
        return

    for sample_id, group in ner_df.groupby("Sample_ID"):
        print(f"📌 Sample {sample_id}: {group['Original_Text'].iloc[0]}")
        for _, row in group.iterrows():
            print(f"   ➝ {row['Entity']:<25} [{row['Label']}]")
        print("-" * 80)


if __name__ == "__main__":
    # Load your CSV
    df = pd.read_csv("/content/drive/MyDrive/ITE Elective 4/NER-Exercise/sample_text.csv")

    # Extract entities
    ner_df = extract_entities(df, text_column="text")

    # Preview results
    preview_entities(ner_df)

    # Save to Excel
    output_file = "/content/drive/MyDrive/ITE Elective 4/NER-Exercise/NER_Results.xlsx"
    ner_df.to_excel(output_file, index=False)
    print(f"\n✅ NER results saved to {output_file}")



--- Extracted Entities (Grouped by Sample) ---

📌 Sample 1: How to Manage Your Personal Brand. Make no mistake: If you have a Facebook account, an Instagram page, a Twitter profile, you are a brand. Every time you upload a photo, add a link, or post an update, you're putting into the world another idea of yourself and what you stand for. It Looks Like Uber's Winning Its War With New York. Grab the popcorn. The Progressive Promise of Today's Technology. A digital policy for the new century, tailored not just to the moment but for the future, is vital if we are to unleash economic growth, shared prosperity, and the full potential of technology for citizens and consumers. But such a policy architecture requires a new consensus -- on privacy, on security, on customer protections, on growth and mobility. U.S. Lawmakers Join Demand For Puerto Rico Governor’s Resignation. Puerto Ricans staged week-long protests calling on Ricardo Rossello to quit over leaked misogynistic and homophobic messa

## **Model Evaluation**

In [ ]:
# Load gold standard (manual annotations)
gold_df = pd.read_excel("/content/drive/MyDrive/ITE Elective 4/NER-Exercise/Gold_Standard.xlsx")

# Load spaCy predictions (from previous step)
pred_df = pd.read_excel("/content/drive/MyDrive/ITE Elective 4/NER-Exercise/NER_Results.xlsx")

# Ensure consistent column names
gold_df = gold_df[["Entity", "Label"]]
pred_df = pred_df[["Entity", "Label"]]


In [ ]:
# Convert both to sets for comparison
gold_set = set([tuple(x) for x in gold_df[["Entity", "Label"]].values])
pred_set = set([tuple(x) for x in pred_df[["Entity", "Label"]].values])

# True Positives (correct matches)
TP = len(gold_set & pred_set)

# False Positives (predicted but not in gold)
FP = len(pred_set - gold_set)

# False Negatives (missed in predictions)
FN = len(gold_set - pred_set)

# Accuracy (on entity level, not token-level)
total_entities = len(gold_set) + len(pred_set - gold_set)  # total unique entities considered
accuracy = TP / total_entities if total_entities > 0 else 0

# Precision, Recall, F1
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n--- NER Performance Metrics ---")
print(f"✅ True Positives (TP): {TP}")
print(f"❌ False Positives (FP): {FP}")
print(f"⚠️ False Negatives (FN): {FN}")
print(f"\n📊 Accuracy : {accuracy:.2f}")
print(f"📊 Precision: {precision:.2f}")
print(f"📊 Recall   : {recall:.2f}")
print(f"📊 F1-Score : {f1_score:.2f}")



--- NER Performance Metrics ---
✅ True Positives (TP): 6
❌ False Positives (FP): 17
⚠️ False Negatives (FN): 15

📊 Accuracy : 0.16
📊 Precision: 0.26
📊 Recall   : 0.29
📊 F1-Score : 0.27
